In [13]:
import polars as pl
import polars.selectors as cs

In [8]:
power_labels = [
    "IBP,",        "ICP,",        "DCP,",      "TCP,",      "CCP,",
    "SHRDP,",      "RFP,",        "INTP,",     "FPUP,",     "DPUP,",
    "INT_MUL24P,", "INT_MUL32P,", "INT_MULP,", "INT_DIVP,", "FP_MULP,",
    "FP_DIVP,",    "FP_SQRTP,",   "FP_LGP,",   "FP_SINP,",  "FP_EXP,",
    "DP_MULP,",    "DP_DIVP,",    "TENSORP,",  "TEXP,",     "SCHEDP,",
    "L2CP,",       "MCP,",        "NOCP,",     "DRAMP,",    "PIPEP,",
    "IDLE_COREP,", "CONSTP",      "STATICP"
]

In [51]:
df = (
    pl.scan_csv(
        "/users/zarand1a/accel-sim-framework/sim_run_12.8/*/NO_ARGS/QV100-Accelwattch_SASS_SIM/accelwattch_power_report.log", 
        has_header=False, 
        separator="?", 
        include_file_paths="path",
    )
    .select(
        pl.col("path").str.extract(r"sim_run_12.8/([^/]*)/.*/accelwattch_power_report.log")
        .alias("benchmark"),
        pl.col("column_1").alias("data")
    )
    .with_row_index()
    .collect()
)

In [80]:
power_df = (
    (
        df.filter(pl.col("data").str.starts_with("gpu_avg_"))
        .select(
            "index",
            pl.col("data")
            .str.replace("^gpu_avg_", "")
            .str.replace_all(" ", "")
            .str.splitn("=", 2)
            .struct.rename_fields(["label", "value"])
            .struct.with_fields(pl.field("value").cast(float))
            .struct.unnest()
        )
    )
    .join_asof(
        df.filter(pl.col("data").str.starts_with("kernel_name"))
        .select(
            pl.col("index"),
            pl.col("index").rank("dense").over("benchmark").alias("serial"),
            pl.col("benchmark"),
            pl.col("data").str.extract("kernel_name = (.*)").alias("kernel")
        ),
        on="index",
    )
    .drop("index")
    .group_by("benchmark", "serial", "kernel", "label").agg(pl.mean("value"))
    .pivot(on="label", index=["benchmark", "serial", "kernel"])
    .select("benchmark", "serial", "kernel", *power_labels)
    .with_columns(
        pl.col("DRAMP,") + pl.col("MCP,"),
        pl.col("L2CP,") + pl.col("NOCP,"),
    )
    .rename(lambda col: col.replace(",", ""))
    .unpivot(index=["benchmark", "serial", "kernel"], value_name="power_w")
    .group_by("benchmark", "serial", "kernel")
    .agg(pl.sum("power_w"))
    .sort("benchmark", "serial", "kernel")
)

In [83]:
df = (
    pl.scan_csv("/users/zarand1a/accel-sim-framework/sim_run_12.8/*/NO_ARGS/QV100-Accelwattch_SASS_SIM/*-NO_ARGS.*",
        has_header=False, 
        separator="&", 
        include_file_paths="path",
    )
    .select(
        pl.col("path").str.extract(r"sim_run_12.8/([^/]*)/.*")
        .alias("benchmark"),
        pl.col("column_1").alias("data")
    )
    .with_row_index()
    .collect()
)

In [89]:
cycle_df = (
    df.filter(pl.col("data").str.starts_with("kernel_name ="))
    .select(
        "index", "benchmark",
        pl.col("data").str.extract("kernel_name = (.*)").alias("kernel"),
    )
).join(
    df.filter(pl.col("data").str.starts_with("gpu_sim_cycle ="))
    .select(
        pl.col("index") - 3,
        pl.col("data").str.extract("gpu_sim_cycle = (.*)").cast(int).alias("cycle"),
    ),
    on="index",
).select(
    "benchmark",
    pl.col("index").rank("dense").over("benchmark").alias("serial"),
    "kernel", "cycle",
)

In [110]:
accel_df = (
    power_df.join(cycle_df, on=["benchmark", "serial", "kernel"])
    .filter(pl.col("power_w").is_not_nan())
    .group_by("benchmark")
    .agg(
        power_w=(pl.col("power_w") * pl.col("cycle")).sum() / pl.col("cycle").sum()
    )
)

In [115]:
df = (
    pl.scan_csv(
        "/proj/threadtune-PG0/amir/power_logs/*-power.0.csv", 
        include_file_paths="path",
    )
    .with_columns(
        pl.col("path").str.extract(r"/proj/threadtune-PG0/amir/power_logs/(.*)-power.0.csv")
        .alias("benchmark"),
    )
    .drop("path")
    .collect()
)

In [116]:
measure_df = (
    df.group_by("benchmark")
    .agg(
        power_w=(
            pl.col("total_energy_millijoules").max() - pl.col("total_energy_millijoules").min()
        ) / (
            pl.col("timestamp_ns").max() - pl.col("timestamp_ns").min()
        ) * 1e6
    )
)

In [119]:
accel_df.rename({"power_w": "accel_power"}).join(measure_df.rename({"power_w": "measure_power"}), on="benchmark")

benchmark,accel_power,measure_power
str,f64,f64
"""gpt2""",92.793675,38.570944
"""bloom""",112.715731,74.344924
"""bert""",96.99283,59.101363
"""resnet50""",92.827203,54.032302
"""deit""",91.044472,54.731851
